# Training the Morphologically-Aware Kapampangan BPE Tokenizer

**Thesis:** *Morphologically-Aware Byte-Pair Encoding Tokenizer for the Kapampangan Language*
— Cansino, Faeldonia, Lucero, Magtanong, Mital (PUP CCIS, 2026)

This notebook runs the **complete tokenizer pipeline** from the thesis, in order,
on Google Colab. Every stage below maps to a specific part of Chapter 3.

---

## First, the honest answer about the GPU

**Training this tokenizer does not use a GPU at all.** It takes about
**15 seconds on a CPU.**

That is not a shortcut — it is what a BPE tokenizer *is*. BPE counts how often
pairs of characters sit next to each other and repeatedly merges the most
frequent pair. It is counting and sorting, not matrix arithmetic. There is
nothing for a GPU to do.

The part of your thesis that *does* need a GPU is the **NLLB-200 translation
experiment** (Chapter 3, p. 44). That is a different stage, and it is currently
blocked for a different reason — explained at the end of this notebook.

So Colab is still useful here: no Python install, no dependency conflicts, and
your teammates can run the identical environment. Just leave the runtime on
**CPU**; selecting a GPU changes nothing and wastes your quota.

---

## What the pipeline does, in one picture

```
   CLEANED_JSON/*.json                UNANNOTATED/*.pdf
   (Kapampangan text)                 (dictionaries)
          |                                  |
          v                                  v
   [A] clean + split 80/10/10        [B] Lexicon Dictionary
          |                            (roots, affixes, variants)
          |                                  |
          +----------------+-----------------+
                           |
                           v
              [C] Pre-Tokenizing        (Figure 6)
                           |
                           v
              [D] Morphological Segmentation   (Figure 7)
                           |
                           v
              [E] Morph-BPE  <- merges may not cross a morpheme boundary
                           |
                           v
              [F] vocabulary-size selection    (p. 43)
                           |
                           v
              [G] the trained tokenizer   (vocab + merge rules, no lexicon)
                           |
                           v
              [H] Fertility / MBF1 / MCF1     (equations 1-7)
```

Run the cells top to bottom. Each one is explained before it runs.

---
# Step 0 — Check the runtime

Confirms Python version and shows that no GPU is present or needed.

In [ ]:
import sys, platform, subprocess
print("Python :", sys.version.split()[0])
print("Machine:", platform.platform())
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10)
    gpu = out.stdout.strip()
    print("GPU    :", gpu if gpu else "none")
except Exception:
    gpu = ""
    print("GPU    : none")
print()
print("A GPU is NOT required for this notebook." if not gpu else
      "A GPU is attached but will not be used: BPE training is pure counting.")

---
# Step 1 — Install dependencies

The tokenizer stages need only four small packages. Colab already has `numpy`
and `pandas`.

| Package | Why it is needed |
|---|---|
| `pypdf` | reads the dictionary PDFs |
| `PyYAML` | reads `config/pipeline.yaml` |
| `regex` | Unicode-aware word splitting for Kapampangan characters (`ñ`, `á`, `û`) |
| `scipy` | the paired statistical tests in Chapter 3 (equations 10-13) |

Deliberately **not** installed here: `torch`, `transformers`, `sacrebleu`.
Those belong to the NLLB translation stage, they are large, and installing them
would suggest this notebook trains a translation model. It does not.

In [ ]:
%pip install -q pypdf PyYAML regex scipy
print("done")

---
# Step 2 — Load the project and your dataset

You need two things in Colab: the **code** (this repository) and the **data**
(your `DATASET` folder).

**Option A — Google Drive (recommended).** Upload the whole
`kapampangan-morphbpe` folder and your `DATASET` folder to Drive once, then
mount it. Nothing is lost when the runtime restarts.

**Option B — Upload a ZIP.** Faster for a one-off run, but you re-upload every
time Colab disconnects.

Run **one** of the two cells below.

In [ ]:
# ---- OPTION A: Google Drive ----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# EDIT these two paths to match where you put the folders in your Drive.
PROJECT_DIR = '/content/drive/MyDrive/kapampangan-morphbpe'
DATASET_DIR = '/content/drive/MyDrive/DATASET'

import os
print("project found:", os.path.isdir(PROJECT_DIR), "->", PROJECT_DIR)
print("dataset found:", os.path.isdir(DATASET_DIR), "->", DATASET_DIR)

In [ ]:
# ---- OPTION B: upload a ZIP ----------------------------------------------
# Zip the kapampangan-morphbpe folder on your PC, run this cell, pick the file.
from google.colab import files
import zipfile, os

up = files.upload()
name = list(up)[0]
with zipfile.ZipFile(name) as z:
    z.extractall('/content')

PROJECT_DIR = '/content/kapampangan-morphbpe'
DATASET_DIR = '/content/kapampangan-morphbpe/../DATASET'   # adjust if needed
print("project found:", os.path.isdir(PROJECT_DIR))

### Point Python at the project

`PYTHONPATH` tells Python where to find the `kapampangan_mt` package. Without
this every later cell fails with `ModuleNotFoundError`.

In [ ]:
import os, sys, subprocess
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
os.environ['PYTHONPATH'] = os.path.join(PROJECT_DIR, 'src')

def run(*args):
    """Run a pipeline script and stream its output.

    Used instead of the `!` shell magic so that paths containing spaces work
    (your Speaking Kapampangan PDF has several) and so a failing stage raises
    instead of scrolling past silently.
    """
    cmd = [sys.executable] + list(args)
    print("$", " ".join(cmd), "\n")
    r = subprocess.run(cmd, cwd=PROJECT_DIR, text=True,
                       env={**os.environ, 'PYTHONPATH': os.path.join(PROJECT_DIR, 'src')})
    if r.returncode != 0:
        raise RuntimeError(f"stage failed with exit code {r.returncode}")

import kapampangan_mt
print("working directory :", os.getcwd())
print("package version   :", kapampangan_mt.__version__)
print("\nfolders present:")
for d in ['src', 'scripts', 'config', 'data', 'docs']:
    print(f"   {d:<10}", "yes" if os.path.isdir(d) else "MISSING")


---
# Stage A — Build the corpus

> **Thesis:** *Sources of Data* (p. 33), *Data Generation and Procedure* (p. 49),
> *Sampling Technique* (p. 34)

### What this does

Reads your Kapampangan text, cleans it, and splits it 80/10/10.

**Exactly which files are used:**

| File | Role |
|---|---|
| `DATASET/CLEANED_JSON/Kapampangan_Religious_Text-1_cleaned.json` | 5,464 Bible verses (Genesis, Deuteronomy, Judges) |
| `DATASET/CLEANED_JSON/Kapampangan_Literary_Text-1_cleaned.json` | 13,310 lines of poetry and prose |

Nothing else feeds the corpus. `dictionary_entries.json` is superseded by
`kapampangan_annotated.json`; both are lexicon sources, not text.

### What the cleaning removes, and why

The literary file was extracted from a legacy `.doc` read as Latin-1, so
compound-document binary leaked into the JSON — you can see the file signature
`ÐÏà¡±á` in the raw text. About **12%** of that file is unusable. The religious
file loses only ~3%, almost all one-word fragments.

Every dropped record is counted **by reason** and written to
`data/raw/ingest_report.json`. Put that table in Chapter 3: you will be asked
how 18,774 raw records became ~17,000 usable lines.

### Why the split is stratified

The thesis (p. 35) requires the split to preserve "major morphological patterns
and sentence-length distributions". A random split would put most Bible prose
(28 words/line) in one part and most poetry (9.8 words/line) in another. The
splitter therefore stratifies by *(length band × morphological complexity ×
domain)* so all three subsets look alike.

**Expect:** ~16,981 lines → 13,584 train / 1,698 validation / 1,699 test.

In [ ]:
run('scripts/10_ingest_dataset.py', '--dataset', DATASET_DIR)
run('scripts/11_split_mono.py')

---
# Stage B — Build the Lexicon Dictionary

> **Thesis:** *Lexicon Dictionary* (p. 41-42)

### Its role, precisely

The Lexicon Dictionary is a **training-time-only** resource. It holds validated
roots, affixes, clitics and spelling variants. It tells the segmenter where the
morpheme joints are — and then it is **thrown away**. The finished tokenizer
never looks at it (p. 42: *"After the tokenizer has been trained, the dictionary
is no longer used during actual input"*).

Think of it as scaffolding: essential while building, absent from the result.

### Where it comes from

| Source file | Contributes |
|---|---|
| `CLEANED_JSON/kapampangan_annotated.json` | 3,138 dictionary headwords with roots and syllables |
| `UNANNOTATED/ilide.info-kapampangandictionaryamongsamson-1-*.pdf` | Samson, 835 pp → 9,763 entries |
| `UNANNOTATED/ilide.info-ka-pampanga-n-*.pdf` | Kapampangan–English–Pilipino wordlist → 3,056 |
| `UNANNOTATED/Speaking Kapampangan -- ... Mirikitani ...pdf` | PALI glossaries → 1,322 |
| `UNANNOTATED/ilide.info-mga-salita-...dayalekto-*.pdf` | 5-language table → 68 |
| `data/lexicon/function_words.tsv` | 61 grammatical morphemes the dictionaries omit |

Three PDFs are **not** extracted, on purpose — they are grammars and papers with
no lexical entries, and mining example words out of prose would violate the rule
that words appearing only inside explanations must not become headwords:
`SL-030-forman-kapampangan-grammar-notes.pdf`,
`ilide.info-an-introduction-to-the-kapampangan-langu-*.pdf`,
`ilide.info-paghahambing-ng-wikang-kapampangan-*.pdf`.

### Three sub-steps

1. **Extract** the PDFs into structured entries (`stage 20`)
2. **Propose** morphology from evidence (`stage 22`)
3. **Merge** the accepted parts into the working lexicon (`stage 24`)

Stage 20 takes ~3 minutes (835 pages). Skip it if `data/extracted/` already
has the files.

In [ ]:
import os
if not os.path.exists('data/extracted/all_entries.jsonl'):
    run('scripts/20_extract_dictionaries.py', '--pdf-dir',
        os.path.join(DATASET_DIR, 'UNANNOTATED'))
else:
    print('data/extracted/ already present - skipping the 3-minute PDF pass')

### Sub-step 2 — propose morphology, from evidence only

> **Thesis:** the Lexicon Dictionary must contain *validated* roots and affixes.

This is where it would be easy to cheat, so here is exactly what happens.

**The tempting shortcut, and why it fails.** Scan the 9,030 Samson headwords for
`long = affix + short` where both are headwords. Run it and the top "prefixes"
are:

```
p-    94 hits    plato = p- + lato
t-    87 hits    tapis = t- + apis
-s   122 hits
```

Those are string coincidences. A lexicon built on them would look plausible and
be worthless.

**What is used instead.** Samson prints paradigms inside its own entries:

> **abut.** Active verb, *mamabut, minabut, mabut*, and its infinitive, *manabut*…

That is the dictionary **asserting** those four forms belong to *abut*. 2,011
entries carry such statements — 6,806 explicit observations. Counting the string
operations across them yields an inventory that looks like Kapampangan:
`i-`, `man-`, `mag-`, `min-`, `in-`, `-an`, `-in-`, `-um-`, `pa-…-an`, `pi-…-an`.

An affix must clear a frequency floor in that explicit set before it may license
any analysis.

**Every entry is `validation_status: "unvalidated"`.** This is a *candidate*
lexicon. A native speaker still has to sign it off.

In [ ]:
import os
if not os.path.exists('data/lexicon_candidate/candidate_lexicon.jsonl'):
    run('scripts/22_build_candidate_lexicon.py')
else:
    print('candidate lexicon already present')
run('scripts/23_qc_candidate_lexicon.py')

### Sub-step 3 — merge into the working lexicon

Two decisions this step makes explicit, both of which belong in Chapter 3.

**Which claims are admitted.** Default is `dictionary_explicit` only — analyses
the dictionary itself prints. Admitting `AI_proposed` means unvalidated machine
guesses shape Morph-BPE training, and a panel member will ask about it.

**Orthography.** Samson uses the Spanish-era system (`acu`, `quing`, `cacu`);
your corpus uses the modern one (`aku`, `king`, `kaku`). Without normalisation
almost none of Samson's headwords match a corpus token, so the merge would add
bulk and no coverage. Both spellings are kept: the normalised form becomes the
root, the original goes to `variants.tsv`.

**One important exclusion.** Dictionaries list *derived* lemmas as headwords
(`sumulat`, `masanting`). Figure 7 checks "is this a root?" **before** trying any
affix rule, so any derived lemma left in `roots.tsv` is returned **unsegmented**
— the tokenizer then learns nothing about its structure. Those are demoted to
`derived_forms.tsv` automatically.

**Expect:** roots 2,642 → ~11,500, and segmenter token coverage 73% → **84%**.

In [ ]:
run('scripts/24_merge_candidate_lexicon.py')

In [ ]:
run('scripts/01_validate_lexicon.py')

---
# Stage C — Pre-Tokenizing

> **Thesis:** *Pre-Tokenizing* (p. 37-38, Figure 6)

### Role

Turns a raw sentence into **word units**, before any morphology is considered.
Three jobs:

1. **Normalise** — Unicode NFC, lowercase, straight quotes, and (optionally)
   spelling rules
2. **Split** on whitespace and punctuation
3. **Mark word starts** with `▁`

### Why the `▁` marker matters

```
"Kinan ne ing pamangan."   ->   ▁kinan   ▁ne   ▁ing   ▁pamangan   .
```

It does two things. Detokenising becomes exact (join the pieces, turn `▁` back
into a space), and a BPE merge can never accidentally glue two words together —
the marker makes a word-initial `k` a different symbol from a word-internal `k`.

### Why normalisation is not cosmetic

Your corpus spells the same word several ways. Measured on your own data:

| One word | Spellings present | Tokens |
|---|---|---|
| `king` | `king`, `qng`, `quing`, `qñg` | 12,007 |
| `ku` | `ku`, `cu`, `cú`, `cû` | 3,420 |
| `karing` | `karing`, `caring`, `caríng`, `cáring` | 2,429 |

Normalising removes **3,769 spurious word types — 14.8% of the vocabulary**.
Without it the tokenizer wastes merges learning four spellings of one word.

The rules live in `data/lexicon/orthography_rules.tsv`. **A Kapampangan
validator must approve each line before you report results.**

Run the cell to see the pre-tokenizer on a real sentence.

In [ ]:
from kapampangan_mt.normalize import Normalizer
from kapampangan_mt.pretokenize import pre_tokenize, words_only

norm = Normalizer.from_rules_file('data/lexicon/orthography_rules.tsv',
                                  lowercase=True, strip_accents=True)

sentence = "Quing kamumulan lelangan na ning Dios ing banua at ing labuad."
print("raw       :", sentence)
print("normalised:", norm.normalize_text(sentence))
print("pre-tokens:", [t.marked for t in pre_tokenize(norm.normalize_text(sentence))])
print()
print("W for the Fertility Rate formula counts only alphabetic words:")
print("   ", words_only(norm.normalize_text(sentence)))

---
# Stage D — Morphological Segmentation

> **Thesis:** *Morphological Segmentation* (p. 38-41) and the **Figure 7**
> pseudocode

### Role

This is the linguistic brain. For each word it tries rules **in the exact order
of Figure 7**, and a rule only fires if what remains is a real root in the
Lexicon Dictionary:

| Order | Rule | Example |
|---|---|---|
| 1 | known compound? | `matuang-lalaki` |
| 2 | bare root? | `kan` |
| 3 | circumfix | `ka-pampang-an` |
| 4 | prefix | `ma-santing` |
| 5 | infix | `k-um-an` |
| 6 | suffix | `sulat-an` |
| 7 | reduplication *(added)* | `ga~gawa` |
| 8 | clitic | `bale-mi` |
| 9 | give up — return the whole word | `eskwela` |

The "remainder must be a real root" test is the safety catch. Without it, `ma` +
anything would look like a valid analysis.

### The thesis example, working

`kan` ("eat") → `k-in-an` (perfective), `k-um-an` (actor focus). The infix sits
*inside* the root, which is exactly what a left-to-right frequency model cannot
see — and it is the case Chapter 1 builds its argument on.

### Two modes

`strict_thesis_mode: true` is a literal transcription of Figure 7.
`strict_thesis_mode: false` (default) is the same order plus five documented
repairs — longest affix first, allomorph resolution, recursive peeling,
reduplication, and stem-final vowel restoration. All five are listed in
`docs/THESIS_ISSUES.md`.

In [ ]:
from kapampangan_mt.lexicon import Lexicon
from kapampangan_mt.segmenter import MorphologicalSegmenter, SegmenterConfig

lex = Lexicon.load('data/lexicon')
seg = MorphologicalSegmenter(lex, normalizer=norm)
print("Lexicon Dictionary:", lex.stats(), "\n")

for w in ["kinan", "kuman", "kan", "sumulat", "sinulat", "sulatan",
          "balemi", "gagawa", "minta", "sinabi", "eskwela"]:
    s = seg.segment(w)
    print(f"  {w:<12} -> {str(s):<44} rule={s.rule:<14} analysed={s.analyzed}")

### Coverage is the number that matters

**Coverage** = the share of words the segmenter can actually analyse. It is the
hard ceiling on your Morpheme Boundary F1, and it is set by how complete the
lexicon is — not by how clever the algorithm is.

Anything the segmenter cannot analyse becomes a single unit for Morph-BPE, which
then behaves like ordinary BPE on that word. That is the intended graceful
degradation, and it is why lexicon completeness matters so much.

In [ ]:
import csv, collections
from kapampangan_mt.pretokenize import words_only

with open('data/processed/train.tsv', encoding='utf-8') as fh:
    train = [r['pam'] for r in csv.DictReader(fh, delimiter='\t')]

freq = collections.Counter(w for s in train for w in words_only(norm.normalize_text(s)))
ok_types = ok_tokens = 0
rules = collections.Counter()
for w, c in freq.items():
    a = seg.segment(w)
    rules[a.rule] += 1
    if a.analyzed:
        ok_types += 1
        ok_tokens += c

print(f"training corpus : {len(train):,} lines, {sum(freq.values()):,} tokens, {len(freq):,} types")
print(f"analysable TYPES : {ok_types/len(freq):.1%}")
print(f"analysable TOKENS: {ok_tokens/sum(freq.values()):.1%}   <- the ceiling on MBF1")
print("\nwhich rule fired, by word type:")
for r, n in rules.most_common():
    print(f"   {r:<16}{n:>7,}")

---
# Stage E — Morph-BPE (the actual training)

> **Thesis:** *Morph-BPE* (p. 42-43), following Asgari et al. (2025)

### Ordinary BPE, in one sentence

Count every pair of adjacent symbols in the corpus, merge the most frequent
pair, repeat until the vocabulary is full.

### What Morph-BPE changes

Each word is stored as a list of **cells**, one per morpheme, and **pairs are
counted only inside a cell**:

```
kapampangan  ->  [▁k a] [p a m p a n g] [a n]
                  ka       pampang        an
```

The pair `(a, p)` straddling `ka|pampang` is invisible to the counter, so the
merge `ap` can never be created from this word. That is the whole mechanism —
one line of difference from Sennrich et al. (2016), and it is what the thesis
is testing.

Words the segmenter could not analyse become a single cell and behave exactly
like ordinary BPE.

### What comes out

Two things, and only these two:

1. a **vocabulary** — specials, 256 byte tokens, the alphabet, and the merged symbols
2. an **ordered merge table**

The Lexicon Dictionary is not part of the output. It has done its job.

**This is the training step. On CPU it takes about 15 seconds.**

### Stage F — choosing the vocabulary size

> **Thesis:** *"The vocabulary size is selected by computing morphological
> distance scores on a validation set"* (p. 43)

The proposal states the criterion but gives no formula, so one is fixed here and
**must be written into your Chapter 3**:

```
MorphDist(V) = mean over validation words of
               1 - |predicted boundaries ∩ gold| / |predicted ∪ gold|

J(V)         = MorphDist(V) + λ · max(0, Fertility(V) − target)
```

The first term asks *do my cuts land on real joints*. The second asks *am I
chopping too finely*. The smallest `V` with the lowest `J` wins.

This runs on the **validation** split only. Selecting `V` on the test split
would leak and invalidate Research Question 1.

In [ ]:
run('scripts/03_train_tokenizer.py')

### Train the control tokenizer

> **Not in the proposal — this is Issue M-3 in `docs/THESIS_ISSUES.md`.**

Research Question 3 asks whether the proposed tokenizer beats NLLB's on Fertility
Rate. It will — **automatically**. NLLB spreads 256,000 tokens across 200
languages, none of them Kapampangan. Any tokenizer trained on Kapampangan wins
that comparison, including one with no morphological awareness whatsoever.

So that comparison tests *"did we train on Kapampangan"*, not *"does morphology
help"*.

The fix is a **matched control**: plain, unconstrained BPE, same corpus, same
vocabulary size, same everything. The only difference is the boundary
constraint. That is the comparison that actually isolates your contribution.

In [ ]:
run('scripts/03_train_tokenizer.py', '--plain-bpe')

---
# Stage G — Runtime behaviour

> **Thesis:** *Runtime tokenization behavior* (p. 43-44)

After training, **no lexicon and no morphological analysis is used**. Encoding a
word is:

1. split into characters (with `▁` fused onto the first one, exactly as in training)
2. any character never seen in training → its UTF-8 **bytes**
3. apply the learned merges greedily, lowest rank first

Byte fallback is why no input is ever unprocessable. The proposal claims
*character* fallback gives this guarantee (p. 44) — it does not, because a
character that never appeared in training has no vocabulary entry. Bytes always
do. This is Issue C-3.

### An honest caveat: the guarantee is statistical, not absolute

Training forbids merges *learned* across a boundary. At runtime there is no
segmentation, so a merge learned inside one word's morpheme can still apply
across a boundary in a different word containing the same letters.

You will see this below: `kinan` may come out as `▁k | inan` rather than
`▁k | in | an`. That is not a bug — it is Issue T-5, and it is worth reporting.
Measure both numbers (segmenter MBF1 = upper bound, runtime MBF1 = what you
deploy); the gap between them is a genuinely interesting result about how much
morphology a merge table can hold.

In [ ]:
from kapampangan_mt.tokenizer import KapampanganTokenizer

morph = KapampanganTokenizer.load('artifacts/tokenizer/morphbpe.json')
plain = KapampanganTokenizer.load('artifacts/tokenizer/plainbpe.json')
print(f"Morph-BPE vocabulary: {morph.vocab_size:,}    plain BPE: {plain.vocab_size:,}\n")

print(f"{'word':<16}{'Morph-BPE':<34}{'plain BPE'}")
print("-" * 84)
for w in ["kinan", "kuman", "sumulat", "sinulat", "sulatan",
          "kapampangan", "pamangan", "balemi", "gagawa", "makapagsulat"]:
    print(f"{w:<16}{' '.join(morph.tokenize_word(w)):<34}{' '.join(plain.tokenize_word(w))}")

s = "Kinan ne ing pamangan king bale."
print("\nfull sentence:", s)
print("  Morph-BPE:", morph.tokenize(s))
print("  round-trip:", repr(morph.decode(morph.encode(s))))

---
# Stage H — Evaluation

> **Thesis:** *Data Analysis* (p. 53-56), equations (1) through (7)

Three tokenizer-level metrics. Each answers a different question.

| Metric | Question | Direction | Equation |
|---|---|---|---|
| **Fertility Rate** | How many pieces per word? | lower = more compact | (1) |
| **Morpheme Boundary F1** | Do my cuts land on real joints? | higher, 1.00 ideal | (2)-(4) |
| **Morphological Consistency F1** | Do words sharing a root share a token? | higher, 1.00 ideal | (5)-(7) |

### Three traps a panel will ask about

**1. Fertility alone is not quality.** A bigger vocabulary always lowers
fertility. That is why the matched plain-BPE control exists. Expect a
**trade-off**: Morph-BPE usually has slightly *higher* fertility and much better
boundary F1 — and the thesis anticipates this (p. 54: fertility counts as better
*"provided that MBF1 and MCF1 are maintained or improved"*).

**2. Morpheme Boundary F1 needs human gold data.** If the "correct" boundaries
come from the same segmenter that constrained training, you are grading a student
with their own answer key. The loader **refuses** machine-generated gold unless
you pass `--allow-auto-gold`, which is for smoke tests only. This is Issue M-2.

**3. Morphological Consistency F1 is undefined until two conventions are fixed.**
If a shared single letter counts as "sharing a token", every pair of words shares
tokens and both tokenizers score near zero. So tokens shorter than 2 characters
are ignored, and "sharing a morpheme" is restricted to **root** morphemes.
Both settings print with the result and **must appear in your Chapter 3** —
results are not comparable across different settings.

In [ ]:
run('scripts/04_eval_tokenizer.py', '--no-nllb', '--allow-auto-gold',
    '--set', 'stats.bootstrap_resamples=200')

In [ ]:
run('scripts/07_report.py')

### The vocabulary-size trade-off curve

This is the single most useful table you can put in Chapter 4. It shows
Morph-BPE beating plain BPE on boundary F1 at **every** vocabulary size, at a
small and consistent fertility cost.

Watch the plain-BPE micro-F1 column: as the vocabulary grows, plain BPE simply
memorises whole frequent words and its boundary score collapses toward zero,
while Morph-BPE holds. That contrast *is* your contribution.

Takes ~2 minutes.

In [ ]:
import csv, json
from collections import Counter
from kapampangan_mt.morph_bpe import MorphBPEConfig, MorphBPETrainer
from kapampangan_mt.tokenizer import BYTE_TOKENS, RESERVED, KapampanganTokenizer
from kapampangan_mt.pretokenize import pre_tokenize
from kapampangan_mt.metrics.boundary_f1 import boundary_f1_corpus, load_gold
from kapampangan_mt.metrics.fertility import fertility_corpus

def col(p, k='pam'):
    with open(p, encoding='utf-8') as fh:
        return [r[k] for r in csv.DictReader(fh, delimiter='\t')]

tr, te = col('data/processed/train.tsv'), col('data/processed/test.tsv')
freqs, segs = Counter(), {}
for s in tr:
    for pt in pre_tokenize(norm.normalize_text(s)):
        freqs[pt.marked] += 1
        if pt.marked not in segs:
            segs[pt.marked] = seg.segment(pt.surface).surfaces if pt.kind == 'word' else [pt.surface]

gold = load_gold('data/gold/morpheme_gold.PROVISIONAL.tsv', allow_auto=True)
reserved = len(RESERVED) + len(BYTE_TOKENS)

print(f"{'V':>7} {'mode':<7} {'fertility':>10} {'MBF1 macro':>11} {'MBF1 micro':>11}")
rows = []
for V in (1000, 2000, 4000, 8000, 16000):
    for constrain, label in ((True, 'morph'), (False, 'plain')):
        m = MorphBPETrainer(MorphBPEConfig(
            vocab_size=V, min_pair_frequency=2,
            constrain_to_morpheme_boundaries=constrain)).train(
            segs, freqs, reserved=reserved, verbose=False)
        tk = KapampanganTokenizer.from_model(m)
        f = fertility_corpus(tk, te); b = boundary_f1_corpus(tk, gold)
        print(f"{V:>7} {label:<7} {f['fertility_corpus']:>10.3f} "
              f"{b['mbf1_macro']:>11.3f} {b['mbf1_micro']:>11.3f}")
        rows.append(dict(V=V, mode=label, fertility=f['fertility_corpus'],
                         mbf1_macro=b['mbf1_macro'], mbf1_micro=b['mbf1_micro']))
json.dump(rows, open('artifacts/results/vocab_sweep.json', 'w'), indent=2)
print("\nsaved -> artifacts/results/vocab_sweep.json")

---
# Step 9 — Download the trained tokenizer

Everything you need to keep. The tokenizer JSON is self-contained: vocabulary +
merge rules, no lexicon required.

In [ ]:
import shutil, os
shutil.make_archive('/content/kapampangan_tokenizer', 'zip', 'artifacts')
print("bundle size:", round(os.path.getsize('/content/kapampangan_tokenizer.zip')/1e6, 2), "MB")
try:
    from google.colab import files
    files.download('/content/kapampangan_tokenizer.zip')
except Exception:
    print("not on Colab — the zip is at /content/kapampangan_tokenizer.zip")

In [ ]:
# Optional: also copy the results back into Drive so they survive the runtime
import shutil, os
if PROJECT_DIR.startswith('/content/drive'):
    dest = os.path.join(PROJECT_DIR, 'artifacts')
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree('artifacts', dest)
    print("copied artifacts/ back to Drive")
else:
    print("project is not on Drive — download the zip above instead")

---
# What this notebook does NOT do, and why

### The NLLB-200 translation experiment is blocked

Research Questions **2** and **4** — BLEU, chrF++, the whole
Kapampangan→Filipino translation half — cannot run, and the reason is data, not
compute.

**There is no parallel data in your dataset.** Every file is monolingual
Kapampangan. The dictionary glosses are **English**, not Filipino. A scan of all
16,981 corpus lines found 4 that look Filipino.

The way out is already in your data: your religious file carries explicit
`Book Chapter:Verse` references for 2,859 verses. Filipino Bibles (*Ang Dating
Biblia*, *Magandang Balita Biblia*) use the same versification. Join on
`(book, chapter, verse)` and you get ~2,800 sentence pairs **mechanically**, no
manual translation. That is days of work, not months.

Once that exists, `scripts/05_train_nmt.py` and `06_translate_eval.py` run — and
*those* need a GPU. Colab's T4 is enough (~45-90 min per arm, three arms).

### The gold standard is still provisional

The evaluation above used `--allow-auto-gold`, which scores against
machine-generated boundaries. **Those numbers are directional only.** For
Chapter 4 you need ~1,000 word types annotated by **two** people independently,
with Cohen's kappa reported. `scripts/02_build_gold_template.py` produces the
form, pre-filled with suggestions so annotators edit rather than type.

### A specific gap in the affix inventory

The affix inventory was derived from Samson's printed paradigms, which are
**verb-focused**. Measured counts for the affixes Kapampangan grammars treat as
central:

| Affix | Explicit observations | Licensed? |
|---|---|---|
| `ma-` | 8 | no (floor is 20) |
| `pa-` | 3 | no |
| `ka-…-an` | 3 | no (floor is 5) |
| `pang-` | 0 | no |

So `masanting` and `kapampangan` currently come back **unsegmented** — the
evidence in this dictionary does not license `ma-` or `ka-…-an`, and inventing
them would break the rule that nothing is asserted without support.

**This is not a bug to code around; it is a task for your validator.** Take the
affix inventory from `SL-030-forman-kapampangan-grammar-notes.pdf` — Forman's
grammar, which is in your `UNANNOTATED/` folder and was deliberately left
unextracted for exactly this reason. Adding `ma-`, `ka-…-an`, `pang-` and its
allomorphs (`pam-`, `pan-`, `panga-`) to `data/lexicon/` will move every number
in Table 2.

---

# Where each result lives

| File | Contents |
|---|---|
| `artifacts/tokenizer/morphbpe.json` | **the trained tokenizer** — vocabulary + merges |
| `artifacts/tokenizer/plainbpe.json` | the matched control tokenizer |
| `artifacts/tokenizer/vocab_search.json` | the vocabulary-size selection table |
| `artifacts/tokenizer/segmentation_train.jsonl` | every training-time analysis (audit trail) |
| `artifacts/results/tokenizer_level.json` | Tables 2 and 4 |
| `artifacts/results/vocab_sweep.json` | the trade-off curve |
| `artifacts/results/REPORT.md` | Tables 2-5 as Markdown, ready to paste |
| `data/raw/ingest_report.json` | what the cleaning dropped, by reason |
| `data/lexicon/MERGE_PROVENANCE.json` | which claims were admitted into the lexicon |

## Next steps, in priority order

1. **Align the Bible verses** against a Filipino Bible → unblocks half the thesis
2. **Fill `data/lexicon/WORKLIST_unanalysed.tsv`** (800 rows, frequency-sorted)
3. **Annotate ~1,000 gold segmentations** with two annotators + kappa
4. **Add the affix inventory from Forman's grammar** — the `ma-` / `ka-…-an` gap above
5. **Get `orthography_rules.tsv` approved** — 8 rules, 30 minutes

Read `docs/THESIS_ISSUES.md` before your defence. It lists 20 problems in the
proposal with explanations and fixes.